# OpenCell, opened cold

This notebook opens the OpenCell store in Arraylake using nothing but the store itself: no source bucket listing, no OpenCell API, no ingest code. Everything below comes from the store's own metadata.

The repo is `earthmover-biology/opencell`. You need an Arraylake login with read access (`al auth login`). The pixels stay in OpenCell's public bucket, `s3://czb-opencell`; the store references them, and Arraylake authorizes those reads.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import zarr
from arraylake import Client

repo = Client().get_repo("earthmover-biology/opencell")
root = zarr.open_group(repo.readonly_session("main").store, mode="r")
about = root.attrs["annotations"]
print(about["title"])
print()
print(about["description"])
print()
print(about["citation"], f"doi:{about['doi']}")
print("licence:", about["license"])

The root says what each top-level tree holds.

In [ ]:
for tree, what in about["trees"].items():
    print(f"{tree}/: {what}")

## Find a protein

The root's `wells` table has one row per well: its path under `plates/`, the tagged gene, OpenCell's cell line ID, the Ensembl ID and the number of fields of view. Each well holds one cell line, and a line's fields can be spread over several plates.

In [ ]:
wells = pd.DataFrame(root.attrs["wells"])
print(f"{len(wells):,} wells, {wells.gene.nunique():,} genes, {wells.fields.sum():,} fields of view")
wells[wells.gene == "AAMP"]

Each well is an OME-Zarr HCS well. Its metadata lists the field images, and its annotations say how the cell line was made.

In [ ]:
well = root[f"plates/{wells[wells.gene == 'AAMP'].image.iloc[0]}"]
fields = [image["path"] for image in well.attrs["ome"]["well"]["images"]]
print(well.path, "holds", fields)
well.attrs["annotations"]

## One field of view

A field image is the confocal z-stack, `(c, z, y, x)`, in micrometres. Its `omero` block names the channels, and its annotations carry the acquisition record and the file every chunk comes from.

In [ ]:
field = well[fields[0]]
ms = field.attrs["ome"]["multiscales"][0]
axes = [a["name"] for a in ms["axes"]]
scale = dict(zip(axes, ms["datasets"][0]["coordinateTransformations"][0]["scale"]))
channels = [c["label"] for c in field.attrs["ome"]["omero"]["channels"]]
stack = field["0"]
print(ms["name"], dict(zip(axes, stack.shape)), stack.dtype)
print("pixel size (µm):", {a: round(s, 4) for a, s in scale.items() if a != "c"})
print("channels:", channels)
print("derived images:", field.attrs["derived_images"])
field.attrs["annotations"]

The root records where the pixel size comes from, because it differs from the value the TIFFs state.

In [ ]:
about["pixel_size_um"]["source"]

The projection OpenCell published is nested in the field as `proj`. Reading it fetches two 600 × 600 planes from the bucket.

In [ ]:
proj = field["proj/0"][:]
extent = [0, stack.shape[-1] * scale["x"], stack.shape[-2] * scale["y"], 0]
fig, axs = plt.subplots(1, 2, figsize=(10, 5), layout="constrained")
for ax, plane, label, cmap in zip(axs, proj, channels, ["Blues_r", "Greens_r"]):
    ax.imshow(plane, cmap=cmap, extent=extent, vmax=np.percentile(plane, 99.8))
    ax.set(title=f"{ms['name']}: {label}", xlabel="x (µm)", ylabel="y (µm)")

A side view through the stack shows the z axis at its true scale. Slicing one row of `y` reads one chunk per plane, so only the planes, not the whole stack, come over the network.

In [ ]:
y = stack.shape[-2] // 2
side = stack[1, :, y, :]
fig, ax = plt.subplots(figsize=(10, 3), layout="constrained")
ax.imshow(side, cmap="Greens_r", vmax=np.percentile(side, 99.8),
          extent=[0, stack.shape[-1] * scale["x"], stack.shape[1] * scale["z"], 0])
ax.set(title=f"{channels[1]}, x–z section at y = {y * scale['y']:.1f} µm", xlabel="x (µm)", ylabel="z (µm)");

The `derived_images` attribute says how `proj` was made. Checking that claim reads the whole stack, 2 × 51 planes for this field, about 73 MB.

In [ ]:
np.array_equal(stack[:].max(axis=1), proj)

## The unlisted cell lines

`unlisted/` is an OME-Zarr collection of the fields whose cell lines the OpenCell website does not list. Its series are paths `<gene>/CID<cell line>/FID<field>`.

In [ ]:
series = root["unlisted/OME"].attrs["ome"]["series"]
genes = {s.split("/")[0] for s in series}
print(f"{len(series)} fields of {len({s.rsplit('/', 1)[0] for s in series})} cell lines, {len(genes)} genes")
series[:3]